In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

**This milestone focuses on formulating the Smart MCQ Solver Challenge as a proper multiple-choice classification problem. You will learn how to convert each prompt and its five options into model-ready inputs, use AutoModelForMultipleChoice to produce logits for A-E, apply LoRA for efficient fine-tuning, and run a small Hugging Face Trainer fine-tuning pipeline.**


In [ ]:
import pandas as pd
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForMultipleChoice,
    TrainingArguments,
    Trainer
)

from datasets import Dataset

from peft import (
    LoraConfig,
    TaskType,
    get_peft_model
)
train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")

MODEL_NAME = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

label_map = {
    "A":0,
    "B":1,
    "C":2,
    "D":3,
    "E":4
}

train["label"] = train["answer"].map(label_map)

**Q1. Label Encoding**
**Convert the answer column in train.csv into numeric labels using the following mapping:**
**A = 0**
**B = 1**
**C = 2**
**D = 3**
**E = 4**

**What is the encoded numeric label for the row at index 150?**


In [ ]:


print(train.loc[150, "label"])

**Q2. Prompt-Option Formatting**
**For row index 0, create the Option B input using exactly this format:**
**str(prompt) + " [SEP] " + str(option_B)**

**What is the exact character length of this formatted input string?**

In [ ]:


formatted = (
    str(train.loc[0, "prompt"])
    + " [SEP] "
    + str(train.loc[0, "B"])
)

print(len(formatted))

**Q3. Single-Row MCQ Tokenization**
**Using bert-base-uncased, tokenize the five formatted inputs for row index 0 with:**
**padding = "max_length"**
**truncation = True**
**max_length = 128**
**return_tensors = "pt"**

**After reshaping for a multiple-choice model, the final input_ids tensor has shape:**
[1, 5, 128]

**What is the value of the second dimension?**

In [ ]:
row = train.loc[0]

choices = [
    str(row["prompt"]) + " [SEP] " + str(row[c])
    for c in ["A","B","C","D","E"]
]

enc = tokenizer(
    choices,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

input_ids = enc["input_ids"].unsqueeze(0)

print(input_ids.shape)
print(input_ids.shape[1])

**Q4. Batch MCQ Tokenization**
**Tokenize the first 16 rows of train.csv as multiple-choice examples.**
**Each row has 5 choices.**
**Each choice is tokenized to length 128.**

**The final input_ids tensor has shape:**
[16, 5, 128]

**How many total token positions are in this tensor?**

In [ ]:
all_input_ids = []

for _, row in train.iloc[:16].iterrows():

    choices = [
        str(row["prompt"]) + " [SEP] " + str(row[c])
        for c in ["A","B","C","D","E"]
    ]

    enc = tokenizer(
        choices,
        padding="max_length",
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )

    all_input_ids.append(enc["input_ids"])

input_ids = torch.stack(all_input_ids)

print(input_ids.shape)

total_positions = input_ids.numel()

print(total_positions)

**Q5. Multiple-Choice Logits**
**Load bert-base-uncased using AutoModelForMultipleChoice.**
**Tokenize row index 0 as 5 choices and pass it through the model.**

**The output logits tensor has shape:**
[1, 5]

**How many logits are produced for one question?**

In [ ]:
model = AutoModelForMultipleChoice.from_pretrained(
    MODEL_NAME
)

row = train.loc[0]

choices = [
    str(row["prompt"]) + " [SEP] " + str(row[c])
    for c in ["A","B","C","D","E"]
]

enc = tokenizer(
    choices,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

outputs = model(
    input_ids=enc["input_ids"].unsqueeze(0),
    attention_mask=enc["attention_mask"].unsqueeze(0)
)

print(outputs.logits.shape)

print(outputs.logits.shape[1])

**Q6. Supervised Loss Tensor**
**For row index 0, pass the tokenized 5-choice input into AutoModelForMultipleChoice along with the correct encoded label.**

**The model returns a scalar loss tensor.**

**How many dimensions does this loss tensor have?**


In [ ]:
label = torch.tensor([train.loc[0, "label"]])

outputs = model(
    input_ids=enc["input_ids"].unsqueeze(0),
    attention_mask=enc["attention_mask"].unsqueeze(0),
    labels=label
)

print(outputs.loss)

print(outputs.loss.dim())

**Q7. LoRA Trainable Parameters**
**Apply LoRA to the bert-base-uncased multiple-choice model using:**
**r = 8**
**lora_alpha = 16**

**target_modules = ["query", "value"]lora_dropout = 0.1bias = "none"task_type = TaskType.SEQ_CLSCount trainable parameters using:sum(p.numel() for p in model.parameters() if p.requires_grad)How many parameters are trainable?***



In [ ]:
!pip install -q peft==0.15.2

In [ ]:
base_model = AutoModelForMultipleChoice.from_pretrained(
    MODEL_NAME
)

config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS
)

model = get_peft_model(base_model, config)

trainable = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(trainable)

**Q8. Hugging Face Dataset Preparation**
**Create a Hugging Face Dataset from the first 100 rows of train.csv.**

**For each row, create:**
**input_ids with shape [5, 128]**
**attention_mask with shape [5, 128]**
**labels as the encoded answer label**

**For the first dataset item, input_ids has shape:
[5, 128]**

**How many tokenized choices are stored in input_ids?**

In [ ]:

small = train.iloc[:100].copy()


def preprocess(example):

    choices = [
        str(example["prompt"]) + " [SEP] " + str(example[c])
        for c in ["A","B","C","D","E"]
    ]

    enc = tokenizer(
        choices,
        padding="max_length",
        truncation=True,
        max_length=128
    )

    return {
        "input_ids": enc["input_ids"],
        "attention_mask": enc["attention_mask"],
        "labels": example["label"]
    }


dataset = Dataset.from_pandas(small)

dataset = dataset.map(preprocess)

print(len(dataset[0]["input_ids"]))

**Q9. Tiny LoRA Fine-Tuning**
**Fine-tune a LoRA multiple-choice model on the first 32 rows using Hugging Face Trainer.**

**Use the following settings:**
**max_length = 64**
**per_device_train_batch_size = 4**
**gradient_accumulation_steps = 1**
**max_steps = 4**

**What is the final global_step reported by the Trainer?**

In [ ]:

small = train.iloc[:32].copy()


def preprocess(example):

    choices = [
        str(example["prompt"]) + " [SEP] " + str(example[c])
        for c in ["A","B","C","D","E"]
    ]

    enc = tokenizer(
        choices,
        padding="max_length",
        truncation=True,
        max_length=64
    )

    return {
        "input_ids": enc["input_ids"],
        "attention_mask": enc["attention_mask"],
        "labels": example["label"]
    }


dataset = Dataset.from_pandas(small)

dataset = dataset.map(preprocess)

dataset.set_format(
    type="torch",
    columns=["input_ids","attention_mask","labels"]
)

base_model = AutoModelForMultipleChoice.from_pretrained(
    MODEL_NAME
)

config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query","value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS
)

model = get_peft_model(base_model, config)


training_args = TrainingArguments(
    output_dir="./mcq_lora",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    max_steps=4,
    logging_steps=1,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset
)

trainer.train()

print(trainer.state.global_step)

**Q10. Probability Assigned to Option E After Fine-Tuning**
**Using the fine-tuned LoRA model from Q9, run inference on row index 0 and apply softmax to the logits.**

**What is the probability assigned to Option E?**

**Round your answer to 4 decimal places.**

In [ ]:
row = train.loc[0]

choices = [
    str(row["prompt"]) + " [SEP] " + str(row[c])
    for c in ["A","B","C","D","E"]
]

enc = tokenizer(
    choices,
    padding="max_length",
    truncation=True,
    max_length=64,
    return_tensors="pt"
)

model.eval()

with torch.no_grad():

    outputs = model(
        input_ids=enc["input_ids"].unsqueeze(0),
        attention_mask=enc["attention_mask"].unsqueeze(0)
    )

probs = torch.softmax(outputs.logits, dim=-1)

print(probs)

print(round(probs[0,4].item(),4))